# Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, accuracy_score

# Load data

In [2]:
df = pd.read_csv('data/imdb/imdb_sentiment.csv')

# Store leaderboard
df_lb = df[df['split'] == 'leaderboard'].copy()
# Overwrite with labeled data only
df = df[df['split'] == 'labeled'].copy()

df.shape

(35000, 4)

In [3]:
df.head()

,id,split,review,sentiment
0,1,labeled,I really liked this Summerslam due to the look...,positive
1,2,labeled,Not many television shows appeal to quite as m...,positive
2,3,labeled,The film quickly gets to a major chase scene w...,negative
3,4,labeled,Jane Austen would definitely approve of this o...,positive
4,5,labeled,Expectations were somewhat high for me when I ...,negative


# Pre-trained Transformers

In [ ]:
pip install transformers

In [ ]:
pip install torch

In [14]:
from transformers import pipeline

In [ ]:
# This model by default predicts positive/negative

model = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    framework='pt'
)

Device set to use cpu


In [ ]:
# Take "raw" (minimum cleaning) reviews, without feature engineering

X = df['review']
y = df['sentiment'].map({'negative': 0, 'positive': 1})

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [ ]:
# Prediction (may take a few minutes)

preds = model(X_test_raw.iloc[:10].values.tolist(), truncation=True)

In [11]:
# Predicted label + probability

preds

[{'label': 'NEGATIVE', 'score': 0.9994311928749084},
 {'label': 'POSITIVE', 'score': 0.9943444728851318},
 {'label': 'POSITIVE', 'score': 0.9966157078742981},
 {'label': 'NEGATIVE', 'score': 0.9005122780799866},
 {'label': 'NEGATIVE', 'score': 0.9995997548103333},
 {'label': 'NEGATIVE', 'score': 0.9998040795326233},
 {'label': 'POSITIVE', 'score': 0.9896134734153748},
 {'label': 'POSITIVE', 'score': 0.9996646642684937},
 {'label': 'POSITIVE', 'score': 0.9943000078201294},
 {'label': 'POSITIVE', 'score': 0.993245542049408}]

In [12]:
y_test_raw.iloc[:10]

17813    0
6857     1
7672     1
9704     0
14303    0
26304    0
3202     1
27310    1
11215    1
20490    1
Name: sentiment, dtype: int64

In [13]:
# Convert labels to 0 or 1 and check test

y_distil_pred_test = pd.DataFrame(preds)['label'].map({'NEGATIVE': 0, 'POSITIVE': 1})
print('Test Accuracy:', accuracy_score(y_test_raw.iloc[:10], y_distil_pred_test))

Test Accuracy: 1.0
